# 03 — Feature engineering

Build a leakage-safe feature table for the hybrid model — every feature for a match uses only earlier matches. Covers Dixon-Coles ratings, internal Elo, recent form, rest, venue, head-to-head, and squad value.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
pd.set_option("display.width", 120)

In [2]:
df = pd.read_parquet("../data/processed/results_clean.parquet")
df["neutral"] = (df["neutral"].astype(str).str.upper()
                 .map({"TRUE": True, "FALSE": False}).fillna(False).astype(bool))
# stable TOTAL order so feature values are reproducible regardless of how the data is sliced
df = df.sort_values(["date","home_team","away_team"], kind="mergesort").reset_index(drop=True)
print(len(df), "matches", df["date"].min().date(), "->", df["date"].max().date())

18989 matches 2006-01-02 -> 2026-06-18


## Dixon-Coles fit

Refit on a training slice; returns ratings + params as a dict.

In [3]:
def fit_dc(df_train, xi=0.0005, maxiter=400):
    teams = sorted(set(df_train["home_team"]) | set(df_train["away_team"]))
    idx = {t: i for i, t in enumerate(teams)}
    n = len(teams)
    hi = df_train["home_team"].map(idx).to_numpy(np.int64)
    ai = df_train["away_team"].map(idx).to_numpy(np.int64)
    hg = df_train["home_score"].to_numpy(np.int64)
    ag = df_train["away_score"].to_numpy(np.int64)
    days_ago = (df_train["date"].max() - df_train["date"]).dt.days.to_numpy(float)
    w = np.exp(-xi * days_ago)
    m00=(hg==0)&(ag==0); m01=(hg==0)&(ag==1); m10=(hg==1)&(ag==0); m11=(hg==1)&(ag==1)
    def unpack(p):
        att = np.concatenate([p[:n-1], [-p[:n-1].sum()]]); dfn = p[n-1:2*n-1]
        return att, dfn, p[2*n-1], p[2*n]
    def tau(lam, mu, rho):
        t = np.ones_like(lam)
        t[m00]=1-lam[m00]*mu[m00]*rho; t[m01]=1+lam[m01]*rho
        t[m10]=1+mu[m10]*rho;          t[m11]=1-rho
        return np.clip(t, 1e-10, None)
    def nll(p):
        att,dfn,ha,rho = unpack(p)
        ll=np.clip(ha+att[hi]+dfn[ai],-10,10); lm=np.clip(att[ai]+dfn[hi],-10,10)
        lam,mu=np.exp(ll),np.exp(lm)
        return -np.sum(w*(hg*ll-lam+ag*lm-mu+np.log(tau(lam,mu,rho))))
    def grad(p):
        att,dfn,ha,rho = unpack(p)
        lam=np.exp(np.clip(ha+att[hi]+dfn[ai],-10,10)); mu=np.exp(np.clip(att[ai]+dfn[hi],-10,10))
        t=tau(lam,mu,rho)
        dl=np.zeros_like(lam); dm=np.zeros_like(lam); dr=np.zeros_like(lam)
        dl[m00]=-mu[m00]*rho; dm[m00]=-lam[m00]*rho; dr[m00]=-lam[m00]*mu[m00]
        dl[m01]=rho; dr[m01]=lam[m01]; dm[m10]=rho; dr[m10]=mu[m10]; dr[m11]=-1.0
        Sl=w*(hg-lam+(lam/t)*dl); Sm=w*(ag-mu+(mu/t)*dm); Sr=w*(dr/t)
        ga=-(np.bincount(hi,Sl,n)+np.bincount(ai,Sm,n)); gd=-(np.bincount(ai,Sl,n)+np.bincount(hi,Sm,n))
        return np.concatenate([ga[:n-1]-ga[n-1], gd, [-Sl.sum()], [-Sr.sum()]])
    x0=np.concatenate([np.zeros(n-1),np.zeros(n),[0.25],[0.0]])
    bounds=[(None,None)]*(2*n-1)+[(None,None),(-0.1,0.1)]
    res=minimize(nll,x0,jac=grad,method="L-BFGS-B",bounds=bounds,options={"maxiter":maxiter})
    att,dfn,ha,rho=unpack(res.x)
    return {"idx":idx,"att":att,"dfn":dfn,"home_adv":float(ha),"rho":float(rho)}

## Static lookups: competitiveness + squad value

Tournament ordinal (friendly → World Cup) and an as-of lookup of each nation's top-25 Transfermarkt value.

In [4]:
def competitiveness(t):
    tl=t.lower()
    if "friendly" in tl: return 0
    if t == "FIFA World Cup": return 5
    if t in {"UEFA Euro","Copa América","African Cup of Nations","AFC Asian Cup","Gold Cup"}: return 4
    if "fifa world cup qualification" in tl: return 3
    if "qualification" in tl or "nations league" in tl: return 2
    return 1

# --- squad value (Transfermarkt, as-of) ---
SQUAD = pd.read_parquet("../data/processed/squad_value.parquet").sort_values("asof_date")
NAME2TM = {"Bosnia and Herzegovina":"Bosnia-Herzegovina","Curaçao":"Curacao",
           "Ivory Coast":"Cote d'Ivoire","Republic of Ireland":"Ireland",
           "South Korea":"Korea, South","North Korea":"Korea, North"}
squad_map = {nat:(g["asof_date"].to_numpy(), g["sv_top25"].to_numpy())
             for nat,g in SQUAD.groupby("nation")}
def squad_value(team, date):
    e = squad_map.get(NAME2TM.get(team, team))
    if e is None: return np.nan
    dts, vals = e
    pos = np.searchsorted(dts, date, side="right") - 1
    return vals[pos] if pos >= 0 else np.nan

print(f"squad-value nations loaded: {SQUAD['nation'].nunique()}")

squad-value nations loaded: 184


## The feature builder

One chronological read-then-update pass: sequential features (Elo, form, rest, H2H) update as it goes; DC ratings refit periodically; venue/squad are per-row lookups.

In [5]:
def build_features(df_in, dc_cadence="6MS", dc_min_train=500):
    d = df_in.sort_values(["date","home_team","away_team"], kind="mergesort").reset_index(drop=True)
    N = len(d)
    dates = d["date"].to_numpy()
    home = d["home_team"].to_numpy(); away = d["away_team"].to_numpy()
    hg = d["home_score"].to_numpy(); ag = d["away_score"].to_numpy()
    neutral = d["neutral"].to_numpy(); host = d["country"].to_numpy()

    # ---------- 1. Dixon-Coles ratings as-of-date (periodic expanding-window refit) ----------
    cuts = pd.date_range(d["date"].min().normalize(), d["date"].max(), freq=dc_cadence)
    fits = {}
    for c in cuts:
        tr = d[d["date"] < c]
        fits[c] = fit_dc(tr) if len(tr) >= dc_min_train else None
    cut_idx = np.searchsorted(cuts.values, dates, side="right") - 1  # latest cut <= date

    att_h=np.full(N,np.nan); dfn_h=np.full(N,np.nan); att_a=np.full(N,np.nan); dfn_a=np.full(N,np.nan)
    dc_lam=np.full(N,np.nan); dc_mu=np.full(N,np.nan)
    dc_ph=np.full(N,np.nan); dc_pd=np.full(N,np.nan); dc_pa=np.full(N,np.nan)
    from scipy.stats import poisson
    gi = np.arange(11)
    for i in range(N):
        ci = cut_idx[i]
        m = fits[cuts[ci]] if ci >= 0 else None
        if m is None: continue
        ix = m["idx"]
        if home[i] not in ix or away[i] not in ix: continue
        ah,dh = m["att"][ix[home[i]]], m["dfn"][ix[home[i]]]
        aa,da = m["att"][ix[away[i]]], m["dfn"][ix[away[i]]]
        adv = 0.0 if neutral[i] else m["home_adv"]; rho = m["rho"]
        lam = np.exp(adv+ah+da); mu = np.exp(aa+dh)
        att_h[i],dfn_h[i],att_a[i],dfn_a[i] = ah,dh,aa,da
        dc_lam[i],dc_mu[i] = lam,mu
        mat = np.outer(poisson.pmf(gi,lam), poisson.pmf(gi,mu))
        mat[0,0]*=1-lam*mu*rho; mat[0,1]*=1+lam*rho; mat[1,0]*=1+mu*rho; mat[1,1]*=1-rho
        s = mat.sum()
        if s > 0:                       # guard: extreme lam/mu can underflow the grid to 0
            mat/=s
            dc_ph[i]=np.tril(mat,-1).sum(); dc_pd[i]=np.trace(mat); dc_pa[i]=np.triu(mat,1).sum()

    # ---------- 2-3-5 sequential pass: Elo, form, rest, head-to-head ----------
    K, HFA = 20.0, 65.0
    elo = {}
    hist = {}            # team -> list of (gf, ga, pts)
    last_played = {}     # team -> date
    h2h = {}             # frozenset(pair) -> list of (home_team, gd)

    elo_h=np.full(N,np.nan); elo_a=np.full(N,np.nan)
    f5_gf_h=np.full(N,np.nan); f5_ga_h=np.full(N,np.nan); f5_ppg_h=np.full(N,np.nan)
    f5_gf_a=np.full(N,np.nan); f5_ga_a=np.full(N,np.nan); f5_ppg_a=np.full(N,np.nan)
    f10_ppg_h=np.full(N,np.nan); f10_ppg_a=np.full(N,np.nan)
    rest_h=np.full(N,np.nan); rest_a=np.full(N,np.nan)
    h2h_n=np.zeros(N); h2h_gd=np.full(N,np.nan); h2h_winrate=np.full(N,np.nan)

    def form(team, n):
        h = hist.get(team, [])
        if not h: return (np.nan,np.nan,np.nan)
        w = h[-n:]
        gf = np.mean([x[0] for x in w]); ga = np.mean([x[1] for x in w]); ppg = np.mean([x[2] for x in w])
        return (gf,ga,ppg)

    for i in range(N):
        H,A = home[i],away[i]
        # --- read (pre-match) ---
        eh = elo.get(H,1500.0); ea = elo.get(A,1500.0)
        elo_h[i]=eh; elo_a[i]=ea
        f5_gf_h[i],f5_ga_h[i],f5_ppg_h[i] = form(H,5)
        f5_gf_a[i],f5_ga_a[i],f5_ppg_a[i] = form(A,5)
        f10_ppg_h[i]=form(H,10)[2]; f10_ppg_a[i]=form(A,10)[2]
        # cap at 7d: intl teams play in synchronized FIFA windows, so only within-week congestion is informative
        if H in last_played: rest_h[i]=min((dates[i]-last_played[H]).astype("timedelta64[D]").astype(int),7)
        if A in last_played: rest_a[i]=min((dates[i]-last_played[A]).astype("timedelta64[D]").astype(int),7)
        key=frozenset((H,A)); pm=h2h.get(key,[])
        if pm:
            h2h_n[i]=len(pm)
            gds=[gd if ht==H else -gd for ht,gd in pm]   # goal diff from current home POV
            h2h_gd[i]=np.mean(gds)
            h2h_winrate[i]=np.mean([1.0 if g>0 else (0.5 if g==0 else 0.0) for g in gds])
        # --- update (post-match) ---
        adv = 0.0 if neutral[i] else HFA
        e_home = 1.0/(1.0+10**((ea-(eh+adv))/400.0))
        s_home = 1.0 if hg[i]>ag[i] else (0.5 if hg[i]==ag[i] else 0.0)
        gd = abs(int(hg[i]-ag[i]))
        g = 1.0 if gd<=1 else (1.5 if gd==2 else (11+gd)/8.0)
        delta = K*g*(s_home-e_home)
        elo[H]=eh+delta; elo[A]=ea-delta
        ph = 3 if hg[i]>ag[i] else (1 if hg[i]==ag[i] else 0)
        pa = 3 if ag[i]>hg[i] else (1 if hg[i]==ag[i] else 0)
        hist.setdefault(H,[]).append((hg[i],ag[i],ph))
        hist.setdefault(A,[]).append((ag[i],hg[i],pa))
        last_played[H]=dates[i]; last_played[A]=dates[i]
        h2h.setdefault(key,[]).append((H,int(hg[i]-ag[i])))

    # ---------- 4. venue (host flag, static per-row) ----------
    is_home_h=np.zeros(N); is_home_a=np.zeros(N)
    for i in range(N):
        H,A=home[i],away[i]
        is_home_h[i]=0.0 if neutral[i] else float(H==host[i])
        is_home_a[i]=0.0 if neutral[i] else float(A==host[i])

    # ---------- 6. squad value (as-of, Transfermarkt) ----------
    sv_h=np.full(N,np.nan); sv_a=np.full(N,np.nan)
    for i in range(N):
        sv_h[i]=squad_value(home[i], dates[i]); sv_a[i]=squad_value(away[i], dates[i])

    out = pd.DataFrame({
        "date":d["date"],"home_team":home,"away_team":away,"neutral":neutral,
        "tournament":d["tournament"],
        "att_home":att_h,"dfn_home":dfn_h,"att_away":att_a,"dfn_away":dfn_a,
        "dc_lam":dc_lam,"dc_mu":dc_mu,"dc_ph":dc_ph,"dc_pd":dc_pd,"dc_pa":dc_pa,
        "elo_home":elo_h,"elo_away":elo_a,"elo_diff":elo_h-elo_a,
        "f5_gf_home":f5_gf_h,"f5_ga_home":f5_ga_h,"f5_ppg_home":f5_ppg_h,
        "f5_gf_away":f5_gf_a,"f5_ga_away":f5_ga_a,"f5_ppg_away":f5_ppg_a,
        "f10_ppg_home":f10_ppg_h,"f10_ppg_away":f10_ppg_a,
        "rest_home":rest_h,"rest_away":rest_a,"rest_diff":rest_h-rest_a,
        "is_home_home":is_home_h,"is_home_away":is_home_a,
        "competitiveness":d["tournament"].map(competitiveness).astype(float),
        "h2h_n":h2h_n,"h2h_gd":h2h_gd,"h2h_winrate":h2h_winrate,
        "sv_home":sv_h,"sv_away":sv_a,"sv_diff":sv_h-sv_a,
        "home_score":hg,"away_score":ag,   # targets
    })
    return out

## Build the full feature table

In [6]:
%time feats = build_features(df)
print(feats.shape)
feats.head(3)

CPU times: user 9.17 s, sys: 107 ms, total: 9.27 s
Wall time: 9.34 s
(18989, 39)


,date,home_team,away_team,neutral,tournament,att_home,dfn_home,att_away,dfn_away,dc_lam,...,is_home_away,competitiveness,h2h_n,h2h_gd,h2h_winrate,sv_home,sv_away,sv_diff,home_score,away_score
0,2006-01-02,Qatar,Libya,False,Friendly,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,125000.0,NaN,NaN,2,0
1,2006-01-05,Egypt,Zimbabwe,False,Friendly,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,3400000.0,NaN,NaN,2,0
2,2006-01-07,Guinea,Togo,True,Friendly,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,9750000.0,8450000.0,1300000.0,1,0


## No-leakage test

Rebuild features on a past-only prefix and confirm every past feature is unchanged — if truncating the future moves a feature, it was peeking ahead.

In [7]:
CUT = pd.Timestamp("2019-12-31")
prefix = df[df["date"] <= CUT].copy()
feats_prefix = build_features(prefix)

feat_cols = [c for c in feats.columns if c not in
             ("date","home_team","away_team","neutral","tournament","home_score","away_score")]

a = feats[feats["date"] <= CUT].reset_index(drop=True)
b = feats_prefix.reset_index(drop=True)
assert len(a) == len(b), (len(a), len(b))

mismatch = {}
for c in feat_cols:
    x, y = a[c].to_numpy(float), b[c].to_numpy(float)
    diff = ~( (np.isnan(x)&np.isnan(y)) | np.isclose(x,y,equal_nan=True,rtol=1e-9,atol=1e-9) )
    if diff.any(): mismatch[c] = int(diff.sum())

if mismatch:
    print("LEAKAGE DETECTED in:", mismatch)
else:
    print(f"PASS — no leakage. {len(feat_cols)} features identical across {len(a)} matches up to {CUT.date()}.")

PASS — no leakage. 32 features identical across 13032 matches up to 2019-12-31.


## Coverage & sanity checks

In [8]:
print("Non-null coverage per feature:")
cov = (1 - feats[feat_cols].isna().mean()).sort_values()
print((cov*100).round(1).to_string())

Non-null coverage per feature:
sv_diff             70.9
h2h_winrate         70.9
h2h_gd              70.9
sv_away             81.1
sv_home             82.1
dc_pa               95.1
dc_ph               95.1
dc_pd               95.1
att_home            95.1
dfn_home            95.1
att_away            95.1
dfn_away            95.1
dc_lam              95.1
dc_mu               95.1
rest_diff           99.1
f5_gf_away          99.4
rest_away           99.4
f5_ga_away          99.4
f5_ppg_away         99.4
f10_ppg_away        99.4
rest_home           99.5
f5_ppg_home         99.5
f5_ga_home          99.5
f5_gf_home          99.5
f10_ppg_home        99.5
is_home_home       100.0
is_home_away       100.0
competitiveness    100.0
h2h_n              100.0
elo_diff           100.0
elo_away           100.0
elo_home           100.0


In [9]:
# Latest Elo top-10 (should look like real football)
last_elo = {}
tmp = feats[["date","home_team","away_team","elo_home","elo_away"]]
# reconstruct final elo from a fresh pass is overkill; instead show strongest by most-recent appearance
recent = (pd.concat([
    feats[["date","home_team","elo_home"]].rename(columns={"home_team":"team","elo_home":"elo"}),
    feats[["date","away_team","elo_away"]].rename(columns={"away_team":"team","elo_away":"elo"}),
]).dropna().sort_values("date").groupby("team").tail(1))
print("Top 10 teams by most-recent pre-match Elo:")
print(recent.sort_values("elo",ascending=False).head(10)[["team","elo"]].to_string(index=False))

Top 10 teams by most-recent pre-match Elo:
       team         elo
  Argentina 2021.158276
      Spain 1993.864350
     Brazil 1956.128028
     France 1937.058889
   Colombia 1918.548442
    England 1898.755183
   Portugal 1897.303787
      Japan 1893.890260
    Morocco 1885.190918
Netherlands 1861.261334


In [10]:
print("rest_days summary (home):"); print(feats["rest_home"].describe().round(1).to_string())
print("\ncompetitiveness counts:"); print(feats["competitiveness"].value_counts().sort_index().to_string())

rest_days summary (home):
count    18885.0
mean         5.5
std          1.8
min          0.0
25%          4.0
50%          7.0
75%          7.0
max          7.0

competitiveness counts:
competitiveness
0.0    6343
1.0    2297
2.0    4470
3.0    4218
4.0    1313
5.0     348


## Save

In [11]:
feats.to_parquet("../data/processed/features.parquet", index=False)
print("wrote ../data/processed/features.parquet", feats.shape)

wrote ../data/processed/features.parquet (18989, 39)
